# 02 · True IEEE 34-bus OPF Case

对比 **Deterministic / CCOPF / Robust** 三种随机最优潮流策略。  
使用**真实 IEEE 34-bus 网络参数**（24.9kV / 1MVA 基准，33 条支路，非均匀阻抗）。

> **PV** 接 Bus 20（原 Bus 834） | **Wind** 接 Bus 9（原 Bus 816）  
> **Robust** 使用 3σ 最坏情况，与 CCOPF（z=1.645）拉开明显差距

In [ ]:
# ── 0. 环境配置（Colab 首次运行）
# !pip install pyomo torch scikit-learn pyyaml seaborn
# !conda install -c conda-forge ipopt -y

In [ ]:
# ── 1. 路径设置 & 导入
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import yaml
import numpy as np
import pandas as pd
from scipy.stats import norm

from src.dataset     import load_and_preprocess_all
from src.model       import QRLSTM
from src.train       import train_model, evaluate
from src.opf_true    import (solve_opf, P_LOAD_TOTAL,
                              PV_BUS, WIND_BUS, SLACK_BUS,
                              P_LOAD_IEEE, TREE_L, V_BASE_KV, Z_BASE)
from src.monte_carlo import run_mc, summarize
from src             import utils

with open('../configs/true_ieee34.yaml') as f:
    cfg = yaml.safe_load(f)

rc, mc_cfg, tr, opf_cfg = cfg['renewable'], cfg['monte_carlo'], cfg['training'], cfg['opf']

print("Config loaded ✓")
print(f"  V_base={V_BASE_KV} kV  |  Z_base={Z_BASE:.1f} Ω")
print(f"  Lines={len(TREE_L)}  |  P_LOAD_TOTAL={P_LOAD_TOTAL:.3f} MVA")
print(f"  PV Bus={PV_BUS}  Wind Bus={WIND_BUS}  Slack Bus={SLACK_BUS}")

## 1 · 可再生能源参数

In [ ]:
# ── 2. 参数设定（从 yaml 读取）
pv_mu      = rc['pv_mu']
wind_mu    = rc['wind_mu']
sigma_pv   = rc['sigma_ratio'] * pv_mu
sigma_wind = rc['sigma_ratio'] * wind_mu
k          = rc['robust_sigma']      # 3σ 最坏情况
pv_low     = max(0.0, pv_mu   - k * sigma_pv)
wind_low   = max(0.0, wind_mu - k * sigma_wind)
pv_high    = pv_mu   + k * sigma_pv
wind_high  = wind_mu + k * sigma_wind

# 理论成本预验证
z05       = norm.ppf(0.95)
slack_det = P_LOAD_TOTAL - (pv_mu + wind_mu)
slack_cc  = P_LOAD_TOTAL - max(0, pv_mu - z05*sigma_pv) - max(0, wind_mu - z05*sigma_wind)
slack_rob = P_LOAD_TOTAL - pv_low - wind_low

print(f"PV  (Bus {PV_BUS:2d}): mu={pv_mu:.3f}  sigma={sigma_pv:.3f}  [{pv_low:.3f}, {pv_high:.3f}] MVA  (±{k}σ)")
print(f"Wind(Bus {WIND_BUS:2d}): mu={wind_mu:.3f}  sigma={sigma_wind:.3f}  [{wind_low:.3f}, {wind_high:.3f}] MVA  (±{k}σ)")
print(f"\n理论成本预验证 (100×p_slack²):")
print(f"  Det={100*slack_det**2:.1f}  CCOPF={100*slack_cc**2:.1f}  Rob={100*slack_rob**2:.1f}")
print(f"  Det < CCOPF < Rob: {100*slack_det**2 < 100*slack_cc**2 < 100*slack_rob**2}")
print(f"  CCOPF/Rob 成本比 = {100*slack_cc**2 / (100*slack_rob**2):.2f}  (目标 < 0.75)")

## 2 · QRLSTM 训练（可选，需要气象数据）

In [ ]:
# ── 3. 训练 QRLSTM（如无数据请跳过此 cell）
DATA_PATTERN = '../data/900131_*.csv'   # ← 修改为你的数据路径

import glob
if glob.glob(DATA_PATTERN):
    X_train, X_test, y_train, y_test, scaler, target_indices = \
        load_and_preprocess_all(DATA_PATTERN, window_size=tr['window_size'])

    model = QRLSTM(X_train.shape[2], hidden_size=tr['hidden_size'])
    model, loss_history = train_model(
        model, X_train, y_train,
        epochs=tr['epochs'], batch_size=tr['batch_size'], lr=tr['lr']
    )
    HAS_MODEL = True
    print("QRLSTM training complete ✓")

    # 从预测结果提取 mu/sigma（实验性，可替代手动设定）
    preds = evaluate(model, X_test, y_test, scaler, target_indices)
    pv_mu_pred   = float(preds['q50'][:, 0].mean())
    wind_mu_pred = float(preds['q50'][:, 1].mean())
    print(f"\nQRLSTM 预测均值: PV={pv_mu_pred:.3f} MW  Wind={wind_mu_pred:.3f} MW")
    print(f"（当前使用 yaml 配置值: PV={pv_mu:.3f}  Wind={wind_mu:.3f}）")
else:
    print("⚠ No data found — skipping QRLSTM training.")
    print("  Put CSV files in data/ and update DATA_PATTERN to enable.")
    HAS_MODEL = False

In [ ]:
# ── 4. 分位数预测可视化（需要已训练模型）
if HAS_MODEL:
    utils.plot_quantile_predictions(model, X_test, y_test, scaler, target_indices)
else:
    print("(Skipped — no model)")

## 3 · OPF 规划求解

In [ ]:
# ── 5. 三模式 OPF 求解
results = {}

print("Solving Deterministic OPF ...")
results['Deterministic'] = solve_opf(
    'det', p_pv_mu=pv_mu, p_wind_mu=wind_mu)

print("Solving CCOPF (ε=0.05) ...")
results['CCOPF'] = solve_opf(
    'ccopf',
    p_pv_mu=pv_mu,       p_wind_mu=wind_mu,
    p_pv_sigma=sigma_pv, p_wind_sigma=sigma_wind,
    epsilon=opf_cfg['epsilon'])

print("Solving Robust OPF (3σ worst case) ...")
results['Robust'] = solve_opf(
    'robust',
    p_pv_mu=pv_mu,    p_wind_mu=wind_mu,
    p_pv_low=pv_low,  p_wind_low=wind_low)

# 汇总表
rows = []
for name, res in results.items():
    rows.append({
        'Mode'         : name,
        'Cost ($)'     : f"{res['cost']:.3f}"    if res['cost']    is not None else 'N/A',
        'p_slack (MVA)': f"{res['p_slack']:.4f}" if res['p_slack'] is not None else 'N/A',
        'Time (s)'     : f"{res['time_s']:.3f}",
        'Status'       : res['status'],
    })
print(pd.DataFrame(rows).set_index('Mode').to_string())
print(f"\n预期: Det($18.7) < CCOPF($48.3) < Rob($83.2)")

In [ ]:
# ── 6. 规划结果可视化
utils.plot_opf_cost(results)

In [ ]:
utils.plot_voltage_profile(results)

## 4 · Monte Carlo 运行验证

In [ ]:
# ── 7. Monte Carlo 验证（纯 NumPy 向量化，500×24 场景 < 1 秒）
#
# 物理依据：
#   needed  = max(0, P_LOAD_TOTAL - pv_real - wind_real)
#   deficit = max(0, needed - cap)
#   不调用 OPF，直接用功率平衡方程，结果等价

mc_stats, caps = run_mc(
    results,
    p_load_total=P_LOAD_TOTAL,
    pv_mu=pv_mu,         wind_mu=wind_mu,
    sigma_pv=sigma_pv,   sigma_wind=sigma_wind,
    n_scenarios=mc_cfg['n_scenarios'],
    n_hours=mc_cfg['n_hours'],
    k_volt=mc_cfg['k_volt'],      # true IEEE34 阻抗更高，k_volt=0.50
    cap_margin=mc_cfg['cap_margin'],
    seed=mc_cfg['seed'],
)
df_summary = summarize(mc_stats)
print("\n预期: Gap Rate  Det~45%  CCOPF~5%  Rob~0%")
print("预期: Avg Cost  Det < CCOPF < Rob")

In [ ]:
# ── 8. MC 可视化（每张图独立 cell，兼容 VS Code）
utils.plot_gap_rate(mc_stats)

In [ ]:
utils.plot_tradeoff(mc_stats)

In [ ]:
utils.plot_voltage_heatmap(mc_stats, method='CCOPF')

In [ ]:
utils.plot_cost_comparison(mc_stats)

In [ ]:
utils.plot_voltage_pdf(mc_stats)

## 5 · 敏感性分析：成本 vs 风险水平 ε

In [ ]:
# ── 9. ε 敏感性分析
epsilons  = [0.001, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.40]
cc_costs  = []

print("Computing CCOPF cost across epsilon values ...")
for eps in epsilons:
    res = solve_opf(
        'ccopf',
        p_pv_mu=pv_mu,       p_wind_mu=wind_mu,
        p_pv_sigma=sigma_pv, p_wind_sigma=sigma_wind,
        epsilon=eps,
    )
    cc_costs.append(res['cost'])
    z = norm.ppf(1 - eps)
    tag = f"{res['cost']:.3f}" if res['cost'] is not None else "N/A"
    print(f"  ε={eps:.3f}  z={z:.3f}  cost={tag}")

In [ ]:
# 敏感性可视化
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

valid_eps   = [epsilons[i] for i, c in enumerate(cc_costs) if c is not None]
valid_costs = [c for c in cc_costs if c is not None]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Cost vs Risk Level ε  (True IEEE 34-bus)", fontsize=13, fontweight='bold')
fig.subplots_adjust(top=0.88, wspace=0.32)

ax = axes[0]
ax.plot(valid_eps, valid_costs, 'o-', color='#4CAF50', linewidth=2.5, markersize=7, label='CCOPF')
ax.axhline(results['Deterministic']['cost'], color='#2196F3', linestyle='--', linewidth=2,
           label=f"Det ${results['Deterministic']['cost']:.1f}")
ax.axhline(results['Robust']['cost'], color='#F44336', linestyle='-.', linewidth=2,
           label=f"Rob ${results['Robust']['cost']:.1f}")
ax.set_xlabel('Risk Level ε', fontsize=11)
ax.set_ylabel('Planning Cost ($)', fontsize=11)
ax.set_title('Cost–Risk Curve', fontsize=11)
ax.legend(fontsize=9); ax.grid(True, linestyle=':', alpha=0.6)
ax.spines[['top','right']].set_visible(False)

ax = axes[1]
ren_totals = [max(0, pv_mu - norm.ppf(1-e)*sigma_pv) + max(0, wind_mu - norm.ppf(1-e)*sigma_wind)
              for e in valid_eps]
ax.plot(valid_eps, ren_totals, 's--', color='#FF9800', linewidth=2, markersize=6,
        label='Trusted renewable (CCOPF)')
ax.axhline(pv_mu + wind_mu,   color='#2196F3', linestyle='--', linewidth=1.5,
           label=f'Det mu={pv_mu+wind_mu:.3f}')
ax.axhline(pv_low + wind_low, color='#F44336', linestyle='-.', linewidth=1.5,
           label=f'Rob low={pv_low+wind_low:.3f}')
ax.set_xlabel('Risk Level ε', fontsize=11)
ax.set_ylabel('Equivalent Renewable Output (MVA)', fontsize=11)
ax.set_title('Mechanism: ε controls trusted renewable', fontsize=11)
ax.legend(fontsize=9); ax.grid(True, linestyle=':', alpha=0.6)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../results/figures/epsilon_tradeoff.png', dpi=150)
plt.show()

## 6 · 保存结果

In [ ]:
# ── 10. 保存指标
utils.save_metrics(results, mc_stats)
print("All done ✓")
print("Figures → results/figures/")
print("Metrics → results/logs/metrics.json")